# Milestone 2: Data Quality dan Profiling

Lanjutkan dari hasil `01_data_profiling.ipynb`. Notebook ini menerapkan profiling DuckDB `SUMMARIZE` dan transformasi Polars untuk menangani missing values, outlier, duplikat, serta validasi data dasar. Sesuaikan nama kolom dan aturan validasi dengan dataset masing-masing.

In [9]:
from pathlib import Path

import duckdb
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / 'data' / 'raw' / 'daily'
DATA_PATH = DATA_DIR / '*.csv'
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'dataset_bersih.parquet'

csv_files = list(DATA_DIR.glob('*.csv'))
if not csv_files:
    raise FileNotFoundError(
        f'Dataset tidak ditemukan di: {DATA_DIR}. '
        'Pastikan Milestone 1 sudah menempatkan data di data/raw/.'
    )

df = pl.scan_csv(
    str(DATA_PATH),
    schema_overrides={
        'open': pl.Float64,
        'low': pl.Float64,
        'high': pl.Float64,
        'close': pl.Float64,
        'volume': pl.Float64,
    },
    ignore_errors=True,
    include_file_paths='file_path',
).with_columns(
    pl.col('file_path')
    .str.split('/')
    .list.last()
    .str.split('\\')
    .list.last()
    .str.replace(r'\.csv$', '')
    .alias('ticker')
).drop('file_path').collect()

print(f'Ukuran awal: {df.shape[0]:,} baris x {df.shape[1]} kolom')
df.head()

Ukuran awal: 2,469,667 baris x 7 kolom


timestamp,open,low,high,close,volume,ticker
str,f64,f64,f64,f64,f64,str
"""2001-04-16""",524.0,524.0,547.0,547.0,342131.0,"""AALI"""
"""2001-04-17""",547.0,524.0,571.0,524.0,345804.0,"""AALI"""
"""2001-04-18""",547.0,524.0,547.0,524.0,230361.0,"""AALI"""
"""2001-04-19""",524.0,500.0,524.0,500.0,457574.0,"""AALI"""
"""2001-04-20""",476.0,452.0,476.0,452.0,228262.0,"""AALI"""


## 1. Profiling dengan DuckDB `SUMMARIZE`

Dapatkan ringkasan statistik seluruh kolom (tipe data, null%, min/max, distinct count) dalam satu perintah.

In [10]:
connection = duckdb.connect()
connection.register('dataset', df)

profil = connection.execute('SUMMARIZE SELECT * FROM dataset').pl()
profil

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""timestamp""","""VARCHAR""","""2001-04-16""","""2023-01-06""",5054,null,null,null,null,null,2469667,0.00
"""open""","""DOUBLE""","""0.0""","""17500000.0""",9078,"""13854.119691440183""","""315497.42820783315""","""122.68384082323429""","""313.9147157892083""","""922.6985617432257""",2469667,0.00
"""low""","""DOUBLE""","""0.0""","""17500000.0""",8944,"""13733.68785427347""","""312561.02662816196""","""120.59437295161896""","""308.7557265705507""","""908.816530120521""",2469667,0.00
"""high""","""DOUBLE""","""0.0""","""17500000.0""",9280,"""14051.004636252579""","""320917.67137429473""","""125.37325979321334""","""319.0158117212029""","""936.5396016860186""",2469667,0.00
"""close""","""DOUBLE""","""1.0""","""17500000.0""",9689,"""13945.465129509363""","""318360.5222451013""","""122.61406347581993""","""314.3237631334229""","""920.8215650274657""",2469667,0.00
"""volume""","""DOUBLE""","""0.0""","""26555346900.0""",553392,"""12418529.026065053""","""88123738.28316598""","""0.0005688643876027284""","""46675.70680049463""","""2435139.163820556""",2469667,0.00
"""ticker""","""VARCHAR""","""AALI""","""ZYRX""",1055,null,null,null,null,null,2469667,0.00


## 2. Deteksi Missing Values

Lihat jumlah dan persentase null per kolom sebelum memutuskan strategi penanganan.

In [11]:
jumlah_baris = len(df)
null_summary = (
    df.null_count()
    .transpose(include_header=True, header_name='kolom', column_names=['jumlah_null'])
    .with_columns(
        (pl.col('jumlah_null') / jumlah_baris * 100).round(2).alias('persen_null')
    )
    .filter(pl.col('jumlah_null') > 0)
    .sort('jumlah_null', descending=True)
)
null_summary

kolom,jumlah_null,persen_null
str,u32,f64


## 3. Penanganan Missing Values

Ganti `kolom_numerik` dan `kolom_kategorik` dengan nama kolom asli pada dataset. Pilih strategi sesuai jenis data: median/mean untuk numerik, nilai default atau modus untuk kategorik.

In [12]:
df_missing_handled = df.with_columns([
    pl.col('open').fill_null(pl.col('open').median()),
    pl.col('low').fill_null(pl.col('low').median()),
    pl.col('high').fill_null(pl.col('high').median()),
    pl.col('close').fill_null(pl.col('close').median()),
    pl.col('volume').fill_null(0.0),
    pl.col('ticker').fill_null('UNKNOWN'),
])

print(f'Null sebelum: {df.null_count().sum_horizontal().sum():,}')
print(f'Null sesudah: {df_missing_handled.null_count().sum_horizontal().sum():,}')

Null sebelum: 0
Null sesudah: 0


## 4. Deteksi dan Penanganan Outlier

Gunakan metode IQR sebagai titik awal, lalu kombinasikan dengan batas bisnis (domain knowledge) yang relevan dengan dataset Anda.

In [13]:
KOLOM_NUMERIK = 'close'

q1 = df_missing_handled[KOLOM_NUMERIK].quantile(0.25)
q3 = df_missing_handled[KOLOM_NUMERIK].quantile(0.75)
iqr = q3 - q1
batas_bawah = max(0.0, q1 - 1.5 * iqr)
batas_atas = q3 + 1.5 * iqr

print(f'Q1={q1}, Q3={q3}, IQR={iqr}')
print(f'Batas bawah: {batas_bawah}, Batas atas: {batas_atas}')

df_outlier_handled = df_missing_handled.with_columns(
    pl.col(KOLOM_NUMERIK).clip(lower_bound=batas_bawah, upper_bound=batas_atas)
)

Q1=122.0, Q3=920.0, IQR=798.0
Batas bawah: 0.0, Batas atas: 2117.0


## 5. Deduplikasi

Tentukan kolom kunci bisnis (misalnya ID unik) sebelum menghapus duplikat — jangan deduplikasi berdasarkan seluruh baris.

In [14]:
KOLOM_KUNCI = ['ticker', 'timestamp']

total_baris = len(df_outlier_handled)
baris_unik = df_outlier_handled.select(KOLOM_KUNCI).n_unique()
print(f'Total baris: {total_baris:,}')
print(f'Baris unik berdasarkan {KOLOM_KUNCI}: {baris_unik:,}')
print(f'Duplikat: {total_baris - baris_unik:,}')

df_dedup = df_outlier_handled.unique(
    subset=KOLOM_KUNCI,
    keep='first',
    maintain_order=True,
)

Total baris: 2,469,667
Baris unik berdasarkan ['ticker', 'timestamp']: 2,469,667
Duplikat: 0


## 6. Validasi Data Dasar

Tulis beberapa aturan validasi sederhana untuk memastikan data bersih memenuhi batas yang masuk akal. Gantikan contoh aturan di bawah dengan aturan yang relevan dengan dataset Anda.

In [15]:
connection.register('dataset_bersih', df_dedup)

validasi = connection.execute(f"""
    SELECT
        COUNT(*) AS total_baris,
        SUM(CASE WHEN {KOLOM_NUMERIK} IS NULL THEN 1 ELSE 0 END) AS pelanggaran_null,
        SUM(CASE WHEN {KOLOM_NUMERIK} < {batas_bawah} OR {KOLOM_NUMERIK} > {batas_atas}
            THEN 1 ELSE 0 END) AS pelanggaran_batas
    FROM dataset_bersih
""").pl()
validasi

total_baris,pelanggaran_null,pelanggaran_batas
i64,"decimal[38,0]","decimal[38,0]"
2469667,0,0


## 7. Simpan Data Bersih

Simpan hasil ke `data/processed/` agar dapat dipakai pada Milestone 3 tanpa mengulang proses cleaning.

In [16]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_dedup.write_parquet(OUTPUT_PATH)

print(f'Baris sebelum cleaning: {len(df):,}')
print(f'Baris sesudah cleaning: {len(df_dedup):,}')
print(f'Data bersih tersimpan di: {OUTPUT_PATH}')

Baris sebelum cleaning: 2,469,667
Baris sesudah cleaning: 2,469,667
Data bersih tersimpan di: D:\UMM\SEMESTER 7\Analysis Big Data\Tugas\tugas1-abd-2026\data\processed\dataset_bersih.parquet


## Catatan Milestone 2

Dokumentasikan di sini: dimensi kualitas data yang paling bermasalah, strategi yang dipilih untuk missing values dan outlier, jumlah baris yang berubah, serta aturan validasi yang diterapkan.

## Catatan Milestone 2

Setelah menjalankan tahapan data cleaning dan validasi kualitas data pada notebook ini, bisa disimpulkan sebagai berikut:

1. **Evaluasi Missing Values**
   - Pengecekan null menunjukkan tidak ditemukan missing value (0 null di seluruh kolom).
   - Fungsi imputasi tetap dijalankan sebagai langkah defensif (defensive programming) untuk memastikan nilai numerik terisi median dan kategorik memiliki nilai default.

2. **Penanganan Outlier dengan IQR & Winsorization**
   - Kolom numerik utama yang dibersihkan adalah harga penutupan (`close`).
   - Dari perhitungan kuartil diperoleh Q1 = 122, Q3 = 920, dan IQR = 798. Batas atas wajar berada di angka 2.117,0, sedangkan batas bawah dipatok ke 0,0 karena harga saham tidak mungkin negatif.
   - Menggunakan teknik Winsorization (clipping), nilai-nilai harga di atas 2.117 dipangkas ke batas atas tanpa menghapus barisnya, sehingga jumlah data tetap utuh dan variasi data menjadi lebih stabil.

3. **Deduplikasi Data**
   - Deduplikasi dilakukan berbasis composite key `['ticker', 'timestamp']` karena satu emiten hanya boleh memiliki satu catatan harga pada tanggal yang sama.
   - Hasil pengecekan menunjukkan tidak ada data duplikat (0 baris ganda dari total 2.469.667 baris).

4. **Validasi Kualitas Data dan Penyimpanan**
   - Validasi query SQL dengan DuckDB membuktikan 0 pelanggaran batas dan 0 pelanggaran null pada data bersih.
   - Data hasil pembersihan berhasil disimpan ke `data/processed/dataset_bersih.parquet` berukuran 16.2 MB.
